In [1]:
import pandas as pd

df = pd.read_csv(r'C:\Users\Dell\OneDrive\Documents\Sales_dashboard_project\train.csv')

print("Data loaded successfully!")
print("Shape:", df.shape)

Data loaded successfully!
Shape: (9800, 18)


In [2]:
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,2,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,3,CA-2017-138688,12/06/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,4,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,5,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680


In [3]:
df.dtypes


Row ID             int64
Order ID             str
Order Date           str
Ship Date            str
Ship Mode            str
Customer ID          str
Customer Name        str
Segment              str
Country              str
City                 str
State                str
Postal Code      float64
Region               str
Product ID           str
Category             str
Sub-Category         str
Product Name         str
Sales            float64
dtype: object

In [4]:
# Fix Order Date - with specific format
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True)

# Fix Postal Code - convert from number to text
df['Postal Code'] = df['Postal Code'].astype(str)

# Confirm the fixes
df.dtypes

Row ID                    int64
Order ID                    str
Order Date       datetime64[us]
Ship Date                   str
Ship Mode                   str
Customer ID                 str
Customer Name               str
Segment                     str
Country                     str
City                        str
State                       str
Postal Code                 str
Region                      str
Product ID                  str
Category                    str
Sub-Category                str
Product Name                str
Sales                   float64
dtype: object

In [5]:
df.isnull().sum()

Row ID            0
Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Customer Name     0
Segment           0
Country           0
City              0
State             0
Postal Code      11
Region            0
Product ID        0
Category          0
Sub-Category      0
Product Name      0
Sales             0
dtype: int64

In [6]:
df['Sales'].describe()


count     9800.000000
mean       230.769059
std        626.651875
min          0.444000
25%         17.248000
50%         54.490000
75%        210.605000
max      22638.480000
Name: Sales, dtype: float64

In [7]:
# Total sales by category
category_sales = df.groupby('Category')['Sales'].sum().sort_values(ascending=False)

print(category_sales)

Category
Technology         827455.8730
Furniture          728658.5757
Office Supplies    705422.3340
Name: Sales, dtype: float64


In [8]:
region_sales = df.groupby('Region')['Sales'].sum().sort_values(ascending=False)
print(region_sales)


Region
West       710219.6845
East       669518.7260
Central    492646.9132
South      389151.4590
Name: Sales, dtype: float64


In [9]:
top_products = df.groupby('Product Name')['Sales'].sum().sort_values(ascending=False).head(10)
print(top_products)


Product Name
Canon imageCLASS 2200 Advanced Copier                                          61599.824
Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind    27453.384
Cisco TelePresence System EX90 Videoconferencing Unit                          22638.480
HON 5400 Series Task Chairs for Big and Tall                                   21870.576
GBC DocuBind TL300 Electric Binding System                                     19823.479
GBC Ibimaster 500 Manual ProClick Binding System                               19024.500
Hewlett Packard LaserJet 3310 Copier                                           18839.686
HP Designjet T520 Inkjet Large Format Printer - 24" Color                      18374.895
GBC DocuBind P400 Electric Binding System                                      17965.068
High Speed Automatic Electric Letter Opener                                    17030.312
Name: Sales, dtype: float64


In [10]:
df['Month'] = df['Order Date'].dt.month
df['Year'] = df['Order Date'].dt.year

Monthly_sales = df.groupby('Month')['Sales'].sum().sort_values(ascending = False)
print(Monthly_sales)

Month
11    350161.7110
12    321480.1695
9     300103.4117
10    199496.2947
3     197573.5872
8     157315.9270
5     154086.7237
6     145837.5233
7     145535.6890
4     136283.0006
1      94291.6296
2      59371.1154
Name: Sales, dtype: float64


In [11]:
# Add month name column for better readability in Tableau
df['Month Name'] = df['Order Date'].dt.strftime('%B')
df['Year'] = df['Order Date'].dt.year

# Export clean data
df.to_csv(r'C:\Users\Dell\OneDrive\Documents\Sales_dashboard_project\clean_sales_data.csv', index=False)

print("Clean data exported successfully!")

Clean data exported successfully!


In [12]:
import sqlite3

# Create a database
conn = sqlite3.connect('superstore.db')

# Load our dataframe into the database as a table
df.to_sql('orders', conn, if_exists='replace', index=False)

print("Database created successfully!")
print("Table 'orders' loaded with", len(df), "rows")


Database created successfully!
Table 'orders' loaded with 9800 rows


In [13]:
# SQL Query 1 - Sales by Category
query1 = """
SELECT Category, 
       ROUND(SUM(Sales), 2) as Total_Sales,
       COUNT(*) as Total_Orders
FROM orders
GROUP BY Category
ORDER BY Total_Sales DESC
"""

result1 = pd.read_sql_query(query1, conn)
print("Sales by Category:")
print(result1)


Sales by Category:
          Category  Total_Sales  Total_Orders
0       Technology    827455.87          1813
1        Furniture    728658.58          2078
2  Office Supplies    705422.33          5909


In [14]:
# SQL Query 2 - Region Performance
query2 = """
SELECT Region,
       ROUND(SUM(Sales), 2) as Total_Sales,
       COUNT(*) as Total_Orders,
       ROUND(AVG(Sales), 2) as Avg_Order_Value
FROM orders
GROUP BY Region
ORDER BY Total_Sales DESC
"""

result2 = pd.read_sql_query(query2, conn)
print("Region Performance:")
print(result2)

Region Performance:
    Region  Total_Sales  Total_Orders  Avg_Order_Value
0     West    710219.68          3140           226.18
1     East    669518.73          2785           240.40
2  Central    492646.91          2277           216.36
3    South    389151.46          1598           243.52


In [15]:
query3 = """
SELECT "Customer Name",
       ROUND(SUM(Sales), 2) as Total_Sales,
       COUNT(*) as Total_Orders,
       ROUND(AVG(Sales), 2) as Avg_Order_Value
FROM orders
GROUP BY "Customer Name"
ORDER BY Total_Sales DESC
LIMIT 10
"""

result3 = pd.read_sql_query(query3, conn)
print("Top 10 Customers:")
print(result3)

Top 10 Customers:
        Customer Name  Total_Sales  Total_Orders  Avg_Order_Value
0         Sean Miller     25043.05            15          1669.54
1        Tamara Chand     19052.22            12          1587.68
2        Raymond Buch     15117.34            18           839.85
3        Tom Ashbrook     14595.62            10          1459.56
4       Adrian Barton     14473.57            20           723.68
5        Ken Lonsdale     14175.23            29           488.80
6        Sanjit Chand     14142.33            22           642.83
7        Hunter Lopez     12873.30            11          1170.30
8        Sanjit Engle     12209.44            19           642.60
9  Christopher Conant     12129.07            11          1102.64


In [16]:
# SQL Query 4 - Monthly Sales Trend
query4 = """
SELECT "Month Name",
       Month,
       ROUND(SUM(Sales), 2) as Total_Sales,
       COUNT(*) as Total_Orders
FROM orders
GROUP BY Month, "Month Name"
ORDER BY Month ASC
"""

result4 = pd.read_sql_query(query4, conn)
print("Monthly Sales Trend:")
print(result4)

Monthly Sales Trend:
   Month Name  Month  Total_Sales  Total_Orders
0     January      1     94291.63           366
1    February      2     59371.12           297
2       March      3    197573.59           680
3       April      4    136283.00           657
4         May      5    154086.72           725
5        June      6    145837.52           691
6        July      7    145535.69           697
7      August      8    157315.93           693
8   September      9    300103.41          1354
9     October     10    199496.29           809
10   November     11    350161.71          1449
11   December     12    321480.17          1382


In [17]:
# SQL Query 5 - Sales by Year
query5 = """
SELECT Year,
       ROUND(SUM(Sales), 2) as Total_Sales,
       COUNT(*) as Total_Orders,
       ROUND(AVG(Sales), 2) as Avg_Order_Value
FROM orders
GROUP BY Year
ORDER BY Year ASC
"""

result5 = pd.read_sql_query(query5, conn)
print("Year over Year Performance:")
print(result5)

Year over Year Performance:
   Year  Total_Sales  Total_Orders  Avg_Order_Value
0  2015    479856.21          1953           245.70
1  2016    459436.01          2055           223.57
2  2017    600192.55          2534           236.86
3  2018    722052.02          3258           221.62


In [18]:
# SQL Query 6 - Sub Category Analysis
query6 = """
SELECT Category,
       "Sub-Category",
       ROUND(SUM(Sales), 2) as Total_Sales,
       COUNT(*) as Total_Orders,
       ROUND(AVG(Sales), 2) as Avg_Order_Value
FROM orders
GROUP BY Category, "Sub-Category"
ORDER BY Total_Sales DESC
LIMIT 10
"""

result6 = pd.read_sql_query(query6, conn)
print("Top 10 Sub-Categories:")
print(result6)

Top 10 Sub-Categories:
          Category Sub-Category  Total_Sales  Total_Orders  Avg_Order_Value
0       Technology       Phones    327782.45           876           374.18
1        Furniture       Chairs    322822.73           607           531.83
2  Office Supplies      Storage    219343.39           832           263.63
3        Furniture       Tables    202810.63           314           645.89
4  Office Supplies      Binders    200028.79          1492           134.07
5       Technology     Machines    189238.63           115          1645.55
6       Technology  Accessories    164186.70           756           217.18
7       Technology      Copiers    146248.09            66          2215.88
8        Furniture    Bookcases    113813.20           226           503.60
9  Office Supplies   Appliances    104618.40           459           227.93
